In [3]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.44SEQ), (0.2023, 0.1994, 0.2010))
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset  = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2)

Ok gotta load the CIFAR-10 dataset, gotta initialize PyTorch, gotta load the dataloaders, gotta set up the model in a class, THEN gotta load it onto the GPU, then gotta train it

See if I can get a loss curve being plotted?

In [38]:
from torch import nn
import torch.nn.functional as F

device = torch.device('cuda:0')

PATCH_SIZE = 4
D = 128
NUM_HEADS = 8
Dk = D // NUM_HEADS

class Net(nn.Module):

    def __init__(self):
        super(Net, self).__init__()
        # Need to construct patches here... 
        self.cls_token = nn.Parameter(torch.randn(1, 1, D))
        self.pos_embeddings = nn.Parameter(torch.randn(1, 65, D)) # 65 is 8x8+1 (number of patches + 1)

        self.fc_embed = nn.Linear(PATCH_SIZE*PATCH_SIZE*3, D)

        # self.layers = nn.ModuleList([
        #     nn.ModuleDict({
        #         "norm1" : nn.LayerNorm(D),
        #         "Wq" : nn.Linear(D, D),
        #         "Wk" : nn.Linear(D, D),
        #         "Wv" : nn.Linear(D, D),
        #         "Wo" : nn.Linear(D, D),
        #         "norm2" : nn.LayerNorm(D),
        #         "fc" : nn.Linear(D, D*4),
        #         "fc2" :nn.Linear(D*4, D) 
        #     })
        #     for _ in range(12)
        # ])

        # Layer 1
        self.norm1 = nn.LayerNorm(D)
        self.Wq_1 = nn.Linear(D, D)
        self.Wk_1 = nn.Linear(D, D)
        self.Wv_1 = nn.Linear(D, D)
        self.Wo_1 = nn.Linear(D, D)
        self.norm1b = nn.LayerNorm(D)
        self.fc1 = nn.Linear(D, D*4)
        self.fc1b = nn.Linear(D*4, D)

        self.fc_final = nn.Linear(D, 10) # 10 output classes
        

    def forward(self, x):
        # x is dim (B, C, H, W)
        B, C, H, W = x.shape

        patches = x.unfold(2, PATCH_SIZE, PATCH_SIZE).unfold(3, PATCH_SIZE, PATCH_SIZE)
        assert patches.shape == (B, 3, 8, 8, 4, 4), patches.shape
        
        x = patches.permute(0, 2, 3, 1, 4, 5).contiguous()
        assert x.shape == (B, 8, 8, 3, 4, 4), x.shape
        
        x = x.view(B, H // PATCH_SIZE, W // PATCH_SIZE, -1)
        assert x.shape == (B, 8, 8, 48), x.shape
                           
        x = x.view(B, -1, PATCH_SIZE*PATCH_SIZE*3) # (B, 64, D)
        assert x.shape == (B, 64, PATCH_SIZE*PATCH_SIZE*3), x.shape
        
        embeddings = self.fc_embed(x)
        assert embeddings.shape == (B, 64, D), embeddings.shape
        cls_tokens = self.cls_token.expand(B, -1, -1) # -1 means keep the dimension as is
        # We use expand here instead of .repeat() so it doesn't copy any memory, just changes teh stride so PyTorch reads teh same (1,DIM) - nice!

        patch_embeddings = torch.cat([cls_tokens, embeddings], dim=1)
        SEQ = patch_embeddings.shape[1]
        assert patch_embeddings.shape == (B, SEQ, D), patch_embeddings.shape

        self.pos_embeddings.expand(B, -1, -1)
        patch_embeddings = patch_embeddings + self.pos_embeddings
        

        ## Layer 1
        # MSA
        x = self.norm1(patch_embeddings) # (B, SEQ, D)
        assert x.shape == (B, SEQ, D), x.shape
        
        q = self.Wq_1(x).view(B, SEQ, NUM_HEADS, Dk).transpose(1,2) # (B, 12, SEQ, 64)
        assert q.shape == (B, NUM_HEADS, SEQ, Dk), x.shape
        
        k = self.Wk_1(x).view(B, x.shape[1], NUM_HEADS, Dk).transpose(1,2)
        v = self.Wv_1(x).view(B, x.shape[1], NUM_HEADS, Dk).transpose(1,2) # (B, 12, SEQ, 64)
        assert v.shape == (B, NUM_HEADS, SEQ, Dk), x.shape
        
        scores = q @ k.transpose(-1, -2) # (B, heads, SEQ, SEQ) # attention pattern per head
        assert scores.shape == (B, NUM_HEADS, SEQ, SEQ), scores.shape
        # no masking
        
        scores = torch.softmax(scores / (Dk ** 0.5), dim=-1) # (B, heads, SEQ, SEQ)
        scaled_v = scores @ v # (B, heads, SEQ, 64)
        assert scaled_v.shape == (B, NUM_HEADS, SEQ, Dk), scaled_v.shape
        
        scaled_v = scaled_v.transpose(1,2).contiguous().view(B, SEQ, D)
        assert scaled_v.shape == (B, SEQ, D), scaled_v.shape
        o = self.Wo_1(scaled_v) # (B, SEQ, D)

        new_embeddings = patch_embeddings + o

        # MLP
        x = self.norm1b(new_embeddings)
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc1b(x)
        
        new_embeddings = new_embeddings + x
        
        cls_embed = new_embeddings[:, 0, :]
        
        return self.fc_final(cls_embed)


net = Net()
net.to(device)


Net(
  (fc_embed): Linear(in_features=48, out_features=128, bias=True)
  (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (Wq_1): Linear(in_features=128, out_features=128, bias=True)
  (Wk_1): Linear(in_features=128, out_features=128, bias=True)
  (Wv_1): Linear(in_features=128, out_features=128, bias=True)
  (Wo_1): Linear(in_features=128, out_features=128, bias=True)
  (norm1b): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (fc1): Linear(in_features=128, out_features=512, bias=True)
  (fc1b): Linear(in_features=512, out_features=128, bias=True)
  (fc_final): Linear(in_features=128, out_features=10, bias=True)
)

In [40]:
from torch import optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=1e-5)

In [41]:
for epoch in range(20):
    running_loss = 0.0
    for i, data in enumerate(train_loader):
        inputs, labels = data[0].to(device), data[1].to(device)
        #inputs, labels = data[0], data[1]

        optimizer.zero_grad()

        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step() # update

        running_loss += loss.item()
        if i % 100 == 99:
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 100:.6f}')
            running_loss = 0.0

print('Finished Training')

[1,   100] loss: 2.452280
[1,   200] loss: 2.351707
[1,   300] loss: 2.279983
[1,   400] loss: 2.253627
[1,   500] loss: 2.234478
[1,   600] loss: 2.218105
[1,   700] loss: 2.204971
[1,   800] loss: 2.187635
[1,   900] loss: 2.173890
[1,  1000] loss: 2.142764
[1,  1100] loss: 2.125400
[1,  1200] loss: 2.126376
[1,  1300] loss: 2.104596
[1,  1400] loss: 2.097602
[1,  1500] loss: 2.081937
[2,   100] loss: 2.060928
[2,   200] loss: 2.078544
[2,   300] loss: 2.033389
[2,   400] loss: 2.037276
[2,   500] loss: 2.028642
[2,   600] loss: 2.018488
[2,   700] loss: 2.026398
[2,   800] loss: 2.023807
[2,   900] loss: 1.986886
[2,  1000] loss: 1.988989
[2,  1100] loss: 1.970244
[2,  1200] loss: 1.971147
[2,  1300] loss: 1.995005
[2,  1400] loss: 1.973230
[2,  1500] loss: 1.965262
[3,   100] loss: 1.946938
[3,   200] loss: 1.927048
[3,   300] loss: 1.933510
[3,   400] loss: 1.951660
[3,   500] loss: 1.940209
[3,   600] loss: 1.913614
[3,   700] loss: 1.940309
[3,   800] loss: 1.918004
[3,   900] l